# Local GPU Smoke Training

This notebook runs the same training pieces as `python -m reconstruction_model.train`, but on a tiny synthetic dataset: real dataloader format, Transformer, AdamW, Muon, cosine schedulers, checkpointing, and W&B logging.

In [2]:
from pathlib import Path
from dataclasses import asdict
from datetime import datetime
import os
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'reconstruction_model').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not find the repository root containing reconstruction_model.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch

from reconstruction_model.dataset import DataConfig, create_dataloaders
from reconstruction_model.model import Transformer, TransformerConfig
from reconstruction_model.schedulers import cosine_scheduler_with_linear_warmup
from reconstruction_model.train import TrainingConfig, set_seed, train
from reconstruction_model.utils import count_model_params
from scripts.smoke_test_training import create_smoke_dataset

assert torch.cuda.is_available(), 'This smoke notebook uses the CUDA training path.'

set_seed(42)
torch.set_float32_matmul_precision('high')
device = torch.device('cuda')
torch.cuda.get_device_name(0)

'NVIDIA L40S'

## Create a tiny training dataset

The generated files use the same layout and file format as the production dataloader expects: `training_data/train/{ER,NR}/traces_energy_*.zst` plus `meta_energy_*.h5`.

In [3]:
work_dir = PROJECT_ROOT / 'artifacts' / 'notebook_smoke'
data_root = work_dir / 'training_data'
cache_root = work_dir / 'cache'
checkpoint_dir = work_dir / 'checkpoints' / datetime.now().strftime('%Y%m%d_%H%M%S')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
cache_root.mkdir(parents=True, exist_ok=True)

trace_samples = 256
create_smoke_dataset(
    data_root,
    n_events_per_recoil=12,
    n_channels=4,
    trace_samples=trace_samples,
    energy=100,
    seed=42,
)

sorted(str(path.relative_to(data_root)) for path in data_root.rglob('*') if path.is_file())

['train/ER/meta_energy_100.h5',
 'train/ER/traces_energy_100.zst',
 'train/NR/meta_energy_100.h5',
 'train/NR/traces_energy_100.zst']

## Build the real dataloaders

This uses `DataConfig` and `create_dataloaders`, so it exercises discovery, metadata loading, `.zst` conversion to the HDF5 cache, train/val splitting, collation, and pinned-memory behavior.

In [4]:
data_config = DataConfig(
    access_mode='local',
    local_data_path=data_root,
    local_cache_path=cache_root,
    max_seq_len=trace_samples,
    recoil_types=['ER', 'NR'],
    train_split=0.75,
    val_split=0.25,
)

batch_size = 2
dataloaders = create_dataloaders(data_config, batch_size=batch_size, num_workers=0)
batch = next(iter(dataloaders['train']))
inputs, spatial_targets, energy_targets, recoil_types = batch
inputs.shape, spatial_targets.shape, energy_targets.shape, recoil_types[:2]

INFO:reconstruction_model.dataset:Found 2 energy levels for train (access_mode: local)
INFO:reconstruction_model.dataset:Initialized train dataset with 18 samples
INFO:reconstruction_model.dataset:Found 2 energy levels for val (access_mode: local)
INFO:reconstruction_model.dataset:Initialized val dataset with 6 samples


Energy 100, Recoil ER: 9 samples for train (out of 12 total)
Energy 100, Recoil NR: 9 samples for train (out of 12 total)
train split energy distribution: {100: 18}

TRAIN SPLIT VERIFICATION:
Energy levels: [100]
Energy distribution: {100: 18}
Recoil distribution: {'ER': 9, 'NR': 9}
Energy 100, Recoil ER: 3 samples for val (out of 12 total)
Energy 100, Recoil NR: 3 samples for val (out of 12 total)
val split energy distribution: {100: 6}

VAL SPLIT VERIFICATION:
Energy levels: [100]
Energy distribution: {100: 6}
Recoil distribution: {'ER': 3, 'NR': 3}
Converting to HDF5: 100keV NR...
  Converting 12 events (filtered from 12)
  Channels: 4, Samples: 256
✓ Converted to HDF5 in 0.0s (0.0 MB)
Converting to HDF5: 100keV ER...
  Converting 12 events (filtered from 12)
  Channels: 4, Samples: 256
✓ Converted to HDF5 in 0.0s (0.0 MB)


(torch.Size([2, 4, 256]), torch.Size([2, 3]), torch.Size([2]), ['NR', 'ER'])

## Initialize the model

The architecture is the same class as production, with tiny dimensions so the full loop finishes quickly on a local GPU.

In [5]:
model_config = TransformerConfig(
    d_model=32,
    d_ff=64,
    max_seq_len=trace_samples,
    patch_len=32,
    patch_stride=32,
    n_head=4,
    n_time_layers=1,
    n_channel_layers=1,
)

model = Transformer(model_config)
model.to_empty(device=device)
model.init_weights()

# Set this to True if you specifically want to smoke-test torch.compile too.
use_compile = False
if use_compile:
    if sys.version_info >= (3, 13):
        raise RuntimeError('torch.compile/Dynamo is not supported on Python 3.13 with torch 2.5.1.')
    model = torch.compile(model)

count_model_params(model, trainable_only=True)

22148

## Optimizers and schedulers

This cell uses the model's production optimizer grouping: AdamW for embeddings, heads, and auxiliary parameters; Muon for matrix-valued transformer block weights.

In [6]:
training_config = TrainingConfig(
    num_steps=2,
    eval_step_period=1,
    save_checkpoint_period=1,
    total_batch_size=batch_size,
    device_batch_size=batch_size,
    num_workers=0,
    checkpoint_dir=checkpoint_dir,
    adamw_fused=True,
    adamw_warmup_steps=1,
    muon_warmup_steps=1,
    wandb_run=True,
    wandb_run_name=f'notebook_smoke_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
    wandb_project_name='DELight_Reconstruction_Smoke',
)

# Use 'online' when you have run `wandb login` locally and want this synced.
os.environ.setdefault('WANDB_MODE', 'offline')

adamw_optimiser, muon_optimiser = model.configure_optimisers(
    adamw_lr=training_config.adamw_lr,
    adamw_betas=training_config.adamw_betas,
    adamw_weight_decay=training_config.adamw_weight_decay,
    adamw_fused=training_config.adamw_fused,
    muon_lr=training_config.muon_lr,
    muon_momentum=training_config.muon_momentum,
    nesterov=training_config.nesterov,
    ns_steps=training_config.ns_steps,
)
adamw_scheduler = cosine_scheduler_with_linear_warmup(
    adamw_optimiser,
    num_warmup_steps=training_config.adamw_warmup_steps,
    total_steps=training_config.num_steps,
)
muon_scheduler = cosine_scheduler_with_linear_warmup(
    muon_optimiser,
    num_warmup_steps=training_config.muon_warmup_steps,
    total_steps=training_config.num_steps,
)

asdict(training_config)

{'num_steps': 2,
 'eval_step_period': 1,
 'save_checkpoint_period': 1,
 'total_batch_size': 2,
 'device_batch_size': 2,
 'num_workers': 0,
 'scalar_loss_weights': (1.0, 1.0),
 'recoil_classification': False,
 'grad_clip': 1.0,
 'checkpoint_dir': PosixPath('/ceph/dwong/transformer/artifacts/notebook_smoke/checkpoints/20260527_222430'),
 'adamw_lr': 0.001,
 'adamw_betas': (0.9, 0.999),
 'adamw_weight_decay': 0.0,
 'adamw_fused': True,
 'muon_lr': 0.001,
 'muon_momentum': 0.95,
 'nesterov': True,
 'ns_steps': 5,
 'adamw_warmup_steps': 1,
 'muon_warmup_steps': 1,
 'wandb_run': True,
 'wandb_run_name': 'notebook_smoke_20260527_222437',
 'wandb_project_name': 'DELight_Reconstruction_Smoke'}

## Train for a few steps

This calls the production `train(...)` function, so the smoke run includes autocast, gradient accumulation, clipping, validation, W&B logging, scheduler stepping, and checkpoint writes.

In [7]:
train(
    model,
    adamw_optimiser,
    adamw_scheduler,
    muon_optimiser,
    muon_scheduler,
    iter(dataloaders['train']),
    iter(dataloaders['val']),
    device,
    training_config,
)

sorted(path.name for path in checkpoint_dir.glob('*.pt'))

INFO:reconstruction_model.train:Step: 0 | Training loss: 11525.6689 | Validation loss: 10916.6250 | Grad norm: 203.2293 | Step duration: 0.55 s
INFO:reconstruction_model.checkpoints:Model checkpoint saved at: /ceph/dwong/transformer/artifacts/notebook_smoke/checkpoints/20260527_222430/reconstruction_model_0.pt
INFO:reconstruction_model.checkpoints:adamw checkpoint saved at: /ceph/dwong/transformer/artifacts/notebook_smoke/checkpoints/20260527_222430/reconstruction_adamw_0.pt
INFO:reconstruction_model.checkpoints:muon checkpoint saved at: /ceph/dwong/transformer/artifacts/notebook_smoke/checkpoints/20260527_222430/reconstruction_muon_0.pt
INFO:reconstruction_model.train:Step: 1 | Training loss: 11785.5400 | Validation loss: 11026.0684 | Grad norm: 205.1366 | Step duration: 0.01 s
INFO:reconstruction_model.checkpoints:Model checkpoint saved at: /ceph/dwong/transformer/artifacts/notebook_smoke/checkpoints/20260527_222430/reconstruction_model_1.pt
INFO:reconstruction_model.checkpoints:adam

Using existing HDF5: energy_100_ER_3013.h5
Using existing HDF5: energy_100_NR_6894.h5


grad_norm,▃█▁
step,▁▅█
step_duration,█▁▁
total_training_time,▁▅█
train_energy_rmse,██▁
train_loss,▆█▁
train_spatial_rmse,▆█▁
val_energy_rmse,█▁▁
val_loss,▁▂█
val_spatial_rmse,▁▂█
grad_norm,202.35368


INFO:reconstruction_model.train:Training finished!


['reconstruction_adamw_0.pt',
 'reconstruction_adamw_1.pt',
 'reconstruction_adamw_2.pt',
 'reconstruction_model_0.pt',
 'reconstruction_model_1.pt',
 'reconstruction_model_2.pt',
 'reconstruction_muon_0.pt',
 'reconstruction_muon_1.pt',
 'reconstruction_muon_2.pt']